# Experiment Tracking and the Model Registry

Notebook for **Chapter 3** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch03.html

Tip: change a number, predict the output, then run the cell again.

## 3.1 Tracking Runs: Parameters, Metrics, Artifacts

In [ ]:
import json, uuid, pathlib, shutil, pickle
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
ROOT = pathlib.Path("/tmp/lab/mini_runs"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir()
class Run:
    def __init__(self, experiment):
        self.id = uuid.uuid4().hex[:8]; self.dir = ROOT/experiment/self.id; self.dir.mkdir(parents=True)
        self.data = {"params": {}, "metrics": {}, "tags": {}}
    def log_param(self, k, v): self.data["params"][k] = v
    def log_metric(self, k, v): self.data["metrics"][k] = v
    def log_artifact(self, name, obj): (self.dir/name).write_bytes(pickle.dumps(obj))
    def end(self): (self.dir/"meta.json").write_text(json.dumps(self.data, indent=1))
def search_runs(experiment, order_by):
    rows = []
    for p in (ROOT/experiment).glob("*/meta.json"):
        d = json.loads(p.read_text()); d["run_id"] = p.parent.name; rows.append(d)
    return sorted(rows, key=lambda r: -r["metrics"][order_by])
X, y = make_classification(n_samples=1500, n_features=15, n_informative=6, flip_y=0.05, random_state=1)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1)
for C in [0.001, 0.01, 0.1, 1, 10]:
    r = Run("churn")
    m = LogisticRegression(C=C, max_iter=500).fit(Xtr, ytr)
    r.log_param("C", C); r.log_metric("test_acc", round(m.score(Xte, yte), 4))
    r.log_metric("train_acc", round(m.score(Xtr, ytr), 4)); r.log_artifact("model.pkl", m); r.end()
print(f"{'run_id':10}{'C':>8}{'train':>8}{'test':>8}")
for row in search_runs("churn", "test_acc"):
    print(f"{row['run_id']:10}{row['params']['C']:>8}{row['metrics']['train_acc']:>8}{row['metrics']['test_acc']:>8}")

In [ ]:
import mlflow, mlflow.sklearn
mlflow.set_experiment("churn")
for C in [0.001, 0.01, 0.1, 1, 10]:
    with mlflow.start_run():
        m = LogisticRegression(C=C, max_iter=500).fit(Xtr, ytr)
        mlflow.log_param("C", C)
        mlflow.log_metric("test_acc", m.score(Xte, yte))
        mlflow.sklearn.log_model(m, "model")      # saves the model as an artifact
# or one line for common libraries:
mlflow.sklearn.autolog()    # logs params, metrics and the model automatically

## 3.2 The Model Registry: Versions, Aliases, Promotion

In [ ]:
registry = {"versions": {}, "aliases": {}}
def register(run_id, acc):
    v = len(registry["versions"]) + 1; registry["versions"][v] = {"run_id": run_id, "acc": acc}; return v
best = search_runs("churn", "test_acc")                  # from the earlier code
v1 = register(best[-1]["run_id"], best[-1]["metrics"]["test_acc"]); registry["aliases"]["champion"] = v1
print("registered v1 (worst run) as champion")
cand = register(best[0]["run_id"], best[0]["metrics"]["test_acc"])
if registry["versions"][cand]["acc"] > registry["versions"][registry["aliases"]["champion"]]["acc"] + 0.005:
    registry["aliases"]["champion"] = cand; print(f"v{cand} beats champion by >0.005 -> promoted")
print(json.dumps(registry["aliases"]))

In [ ]:
result = mlflow.register_model(f"runs:/{run_id}/model", "churn")       # creates a new version
from mlflow import MlflowClient
client = MlflowClient()
client.set_registered_model_alias("churn", "champion", result.version)  # move the label
model = mlflow.pyfunc.load_model("models:/churn@champion")              # serving loads by alias

## 3.3 Hyperparameter Search and Cost Control

In [ ]:
import random
from sklearn.ensemble import RandomForestClassifier
rng = random.Random(0); best = (0, None)
space = {"n_estimators": [20, 50, 100, 200], "max_depth": [2, 4, 8, 16, None], "min_samples_leaf": [1, 2, 5, 10]}
print("full grid size:", 4*5*4)
for t in range(12):
    p = {k: rng.choice(v) for k, v in space.items()}
    acc = RandomForestClassifier(random_state=0, **p).fit(Xtr, ytr).score(Xva, yva)
    if acc > best[0]: best = (acc, p)
print("best:", best)

In [ ]:
import optuna
def objective(trial):
    depth = trial.suggest_int("max_depth", 2, 16)
    n = trial.suggest_int("n_estimators", 20, 200)
    return RandomForestClassifier(max_depth=depth, n_estimators=n, random_state=0).fit(Xtr, ytr).score(Xva, yva)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

## 3.4 Hyperparameter Search at Scale: Random, Halving and Bayesian

Section 3.3 introduced search and cost control. This section shows the methods you will meet in real tools (Optuna, Ray Tune, SageMaker and Vertex AI tuning) and how to reason about their cost. Everything below that has a printed output was run here with numpy and scikit-learn. Optuna is not installed here, so the Optuna script is labelled not run here. I used a hand-written successive-halving loop to demonstrate the idea it implements.

In [ ]:
import numpy as np
# A made-up but realistic objective: learning rate matters a lot, dropout barely matters.
def score(lr, dropout):
    return np.exp(-((np.log10(lr) + 2.3) ** 2) / 0.5) - 0.02 * (dropout - 0.3) ** 2

def best_grid():
    lrs = np.logspace(-4, -1, 3)            # 3 x 3 = 9 trials
    drops = np.linspace(0.0, 0.6, 3)
    return max(score(a, b) for a in lrs for b in drops)

def best_random(rng):
    lr = 10 ** rng.uniform(-4, -1, 9)       # 9 trials, every one tries a new learning rate
    dr = rng.uniform(0.0, 0.6, 9)
    return max(score(a, b) for a, b in zip(lr, dr))

rng = np.random.default_rng(0)
rand = [best_random(rng) for _ in range(2000)]
print("grid, 9 trials:           best score", round(best_grid(), 3))
print("random, 9 trials (mean of 2000 repeats):", round(float(np.mean(rand)), 3))
print("random beats grid in", f"{np.mean(np.array(rand) > best_grid()):.0%}", "of repeats")
print("distinct learning rates tried: grid 3, random 9")

grid, 9 trials:           best score 0.923
random, 9 trials (mean of 2000 repeats): 0.93
random beats grid in 71% of repeats
distinct learning rates tried: grid 3, random 9


In [ ]:
# Budget arithmetic for successive halving (reduction factor 3)
n, r, eta = 27, 1, 3
total = 0
while n >= 1:
    print(f"{n:2d} trials x {r:2d} epochs = {n * r:3d}")
    total += n * r
    n //= eta; r *= eta
print("total", total, "versus", 27 * 27, "for full training of all")

27 trials x  1 epochs =  27
 9 trials x  3 epochs =  27
 3 trials x  9 epochs =  27
 1 trials x 27 epochs =  27
total 108 versus 729 for full training of all


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=6000, n_features=30, n_informative=8, flip_y=0.08, random_state=3)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=3)
rng = np.random.default_rng(5)
configs = [{"alpha": 10 ** rng.uniform(-6, -1), "eta0": 10 ** rng.uniform(-4, -0.5)} for _ in range(27)]

def make(cfg):
    return SGDClassifier(loss="log_loss", alpha=cfg["alpha"], learning_rate="constant", eta0=cfg["eta0"], random_state=0)

def train_epochs(model, n):
    for _ in range(n):
        model.partial_fit(Xtr, ytr, classes=[0, 1])
    return roc_auc_score(yva, model.decision_function(Xva))

# Method A: train all 27 configs for the full 27 epochs
full = [train_epochs(make(c), 27) for c in configs]
print("A) all configs, 27 epochs: cost", 27 * 27, "epochs, best AUC", round(max(full), 4))

# Method B: successive halving: 27 configs at 1 epoch, keep best third, give them 3x more
models = [make(c) for c in configs]; epochs_done = 0; cost = 0
alive = list(range(27)); budget = 1
while True:
    scores = {}
    for i in alive:
        extra = budget - epochs_done
        scores[i] = train_epochs(models[i], extra)
        cost += extra
    epochs_done = budget
    print(f"   rung: {len(alive):2d} configs, {budget:2d} epochs each, best AUC so far {max(scores.values()):.4f}")
    if len(alive) == 1: break
    keep = max(1, len(alive) // 3)
    alive = sorted(alive, key=lambda i: -scores[i])[:keep]
    budget *= 3
print("B) successive halving: cost", cost, "epochs, final AUC", round(scores[alive[0]], 4))
print("best config found by A is also the winner of B:", int(np.argmax(full)) == alive[0])

A) all configs, 27 epochs: cost 729 epochs, best AUC 0.8248
   rung: 27 configs,  1 epochs each, best AUC so far 0.8256
   rung:  9 configs,  3 epochs each, best AUC so far 0.8254
   rung:  3 configs,  9 epochs each, best AUC so far 0.8245
   rung:  1 configs, 27 epochs each, best AUC so far 0.8245
B) successive halving: cost 81 epochs, final AUC 0.8245
best config found by A is also the winner of B: False


In [ ]:
import optuna
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=6000, n_features=30, n_informative=8, flip_y=0.08, random_state=3)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=3)

def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-6, 1e-1, log=True)
    eta0 = trial.suggest_float("eta0", 1e-4, 0.3, log=True)
    model = SGDClassifier(loss="log_loss", alpha=alpha, learning_rate="constant", eta0=eta0, random_state=0)
    for epoch in range(27):
        model.partial_fit(Xtr, ytr, classes=[0, 1])
        auc = roc_auc_score(yva, model.decision_function(Xva))
        trial.report(auc, epoch)              # tell Optuna how this trial is doing
        if trial.should_prune():              # a clearly losing trial stops early
            raise optuna.TrialPruned()
    return auc

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=0),
                            pruner=optuna.pruners.HyperbandPruner())
study.optimize(objective, n_trials=60)
print(study.best_params, study.best_value)

## Chapter recap

## Practice set

Problem 3.1 Name three things a tracker should store besides the metric. **Answer:** Parameters, artifacts (the model), and code version, data version, environment or seed.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch03.html